This file is to test if my functions are working!

Developing them here, and then pulling from elsewhere.

Data preprocessing first

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load_heart_disease_data(filepath):
    """
    Load the heart disease dataset from CSV.
    
    Parameters
    ----------
    filepath : str
        Path to the heart disease CSV file
        
    Returns
    -------
    pd.DataFrame
        Raw dataset with all features and targets
        
    Raises
    ------
    FileNotFoundError
        If the CSV file does not exist
    ValueError
        If the CSV is empty or malformed
        
    Examples
    --------
    >>> df = load_heart_disease_data('data/heart_disease_uci.csv')
    >>> df.shape
    (270, 15)
    """
    # Hint: Use pd.read_csv()
    # Hint: Check if file exists and raise helpful error if not
    # TODO: Implement data loading
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        raise FileNotFoundError(f"The file {filepath} was not found. Please check the path and try again.")
    except ValueError:
        raise ValueError(f"The file {filepath} is not a valid CSV. Please check the file format and try again.")

    return df

data = load_heart_disease_data(r'C:\Users\mayac\BINF5507_a2\assignment-2-supervised-learning-mclappe\data\heart_disease_uci.csv')

data.head()

,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


Now testing if that works on its own

In [4]:
import sys
sys.path.append(r'C:\Users\mayac\BINF5507_a2\assignment-2-supervised-learning-mclappe\students')
from data_processing import load_heart_disease_data

In [5]:
data = load_heart_disease_data(r'C:\Users\mayac\BINF5507_a2\assignment-2-supervised-learning-mclappe\data\heart_disease_uci.csv')

Next step - preprocess data

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def preprocess_data(df):
    """
    Handle missing values, encode categorical variables, and clean data.
    
    Parameters
    ----------
    df : pd.DataFrame
        Raw dataset
        
    Returns
    -------
    pd.DataFrame
        Cleaned and preprocessed dataset
    """
    # TODO: Implement preprocessing
    # - Handle missing values
    # - Encode categorical variables (e.g., sex, cp, fbs, etc.)
    # - Ensure all columns are numeric

    categorical_cols = [
        "sex", "dataset", "cp", "fbs", "restecg", "exang", "slope", "thal"]
    numeric_cols = df.columns.difference(categorical_cols)

    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
    
    for col in categorical_cols:
        df[col] = df[col].fillna(df[col].mode()[0])
                                               
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

    return df

preprocesseddata = preprocess_data(data)

In [12]:
def prepare_regression_data(df, target='chol'):
    """
    Prepare data for linear regression (predicting serum cholesterol).
    
    Parameters
    ----------
    df : pd.DataFrame
        Preprocessed dataset
    target : str
        Target column name (default: 'chol')
        
    Returns
    -------
    tuple
        (X, y) feature matrix and target vector
    """
    # TODO: Implement regression data preparation
    # - Remove rows with missing chol values
    # - Exclude chol from features
    # - Return X (features) and y (target)

    df = df.dropna(subset=[target])
    y = df[target]
    X = df.drop(columns=[target])
    
    return X, y

X_reg, y_reg = prepare_regression_data(preprocesseddata)

In [13]:
def prepare_classification_data(df, target='num'):
    """
    Prepare data for classification (predicting heart disease presence).
    
    Parameters
    ----------
    df : pd.DataFrame
        Preprocessed dataset
    target : str
        Target column name (default: 'num')
        
    Returns
    -------
    tuple
        (X, y) feature matrix and target vector (binary)
    """
    # TODO: Implement classification data preparation
    # - Binarize target variable
    # - Exclude target from features
    # - Exclude chol from features
    # - Return X (features) and y (target)
    y = (df[target] > 0).astype(int)
    X = df.drop(columns=[target, 'chol'])

    return X, y

X_class, y_class = prepare_classification_data(preprocesseddata)

In [14]:
def split_and_scale(X, y, test_size=0.2, random_state=42):
    """
    Split data into train/test sets and scale features.
    
    Parameters
    ----------
    X : pd.DataFrame or np.ndarray
        Feature matrix
    y : pd.Series or np.ndarray
        Target vector
    test_size : float
        Proportion of data to use for testing
    random_state : int
        Random seed for reproducibility
        
    Returns
    -------
    tuple
        (X_train_scaled, X_test_scaled, y_train, y_test, scaler)
        where scaler is the fitted StandardScaler
    """
    # TODO: Implement train/test split and scaling
    # - Use train_test_split with provided parameters
    # - Fit StandardScaler on training data only
    # - Transform both train and test data
    # - Return scaled data and scaler object

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test, scaler

X_train_scaled, X_test_scaled, y_train, y_test, scaler = split_and_scale(X_class, y_class)

Now we're doing regression

In [22]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score

def train_elasticnet_grid(X_train, y_train, l1_ratios, alphas):
    """
    Train ElasticNet models over a grid of hyperparameters.
    
    Parameters
    ----------
    X_train : np.ndarray or pd.DataFrame
        Training feature matrix
    y_train : np.ndarray or pd.Series
        Training target vector
    l1_ratios : list or np.ndarray
        L1 ratio values to test (0 = L2 only, 1 = L1 only)
    alphas : list or np.ndarray
        Regularization strength values to test
        
    Returns
    -------
    pd.DataFrame
        DataFrame with columns: ['l1_ratio', 'alpha', 'r2_score', 'model']
        Contains R² scores for each parameter combination on training data
    """
    # TODO: Implement grid search
    # - Create results list
    # - For each combination of l1_ratio and alpha:
    #   - Train ElasticNet model with max_iter=5000
    #   - Calculate R² score on training data
    #   - Store results
    # - Return DataFrame with results

    results = []
    for l1_ratio in l1_ratios:
        for alpha in alphas:
            model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=5000, random_state=42)
            model.fit(X_train, y_train)
            r2 = r2_score(y_train, model.predict(X_train))
            results.append({'l1_ratio': l1_ratio, 'alpha': alpha, 'r2_score': r2, 'model': model})
    return pd.DataFrame(results)

enet_grid = train_elasticnet_grid(X_train_scaled, y_train, l1_ratios=[0.3, 0.5, 0.7], alphas=[0.01, 0.1, 1.0])
enet_grid.head()

,l1_ratio,alpha,r2_score,model
0,0.3,0.01,0.508995,"ElasticNet(alpha=0.01, l1_ratio=0.3, max_iter=..."
1,0.3,0.10,0.472288,"ElasticNet(alpha=0.1, l1_ratio=0.3, max_iter=5..."
2,0.3,1.00,0.000000,"ElasticNet(l1_ratio=0.3, max_iter=5000, random..."
3,0.5,0.01,0.508361,"ElasticNet(alpha=0.01, max_iter=5000, random_s..."
4,0.5,0.10,0.433767,"ElasticNet(alpha=0.1, max_iter=5000, random_st..."


In [21]:
#heatmap
def create_r2_heatmap(results_df, l1_ratios, alphas, output_path=None):
    """
    Create a heatmap of R² scores across l1_ratio and alpha parameters.
    
    Parameters
    ----------
    results_df : pd.DataFrame
        Results from train_elasticnet_grid
    l1_ratios : list or np.ndarray
        L1 ratio values used in grid
    alphas : list or np.ndarray
        Alpha values used in grid
    output_path : str, optional
        Path to save figure. If None, returns figure object
        
    Returns
    -------
    matplotlib.figure.Figure
        The heatmap figure
    """
    # TODO: Implement heatmap creation
    # - Pivot results_df to create matrix with l1_ratio on x-axis, alpha on y-axis
    # - Create heatmap using seaborn
    # - Set labels: "L1 Ratio", "Alpha", "R² Score"
    # - Add colorbar
    # - Save to output_path if provided
    # - Return figure object

    pivot_df = results_df.pivot(index='alpha', columns='l1_ratio', values='r2_score')
    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot_df, annot=True, fmt=".3f", cmap='viridis', cbar_kws={'label': 'R² Score'})
    plt.xlabel('L1 Ratio')
    plt.ylabel('Alpha')

    if output_path:
        plt.savefig(output_path)
        plt.close()
    return plt.gcf()

create_r2_heatmap(enet_grid, l1_ratios=[0.3, 0.5, 0.7], alphas=[0.01, 0.1, 1.0], output_path='enet_r2_heatmap.png')

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

In [24]:
#best pick
def get_best_elasticnet_model(X_train, y_train, X_test, y_test, 
                               l1_ratios=None, alphas=None):
    """
    Find and train the best ElasticNet model on test data.
    
    Parameters
    ----------
    X_train : np.ndarray or pd.DataFrame
        Training features
    y_train : np.ndarray or pd.Series
        Training target
    X_test : np.ndarray or pd.DataFrame
        Test features
    y_test : np.ndarray or pd.Series
        Test target
    l1_ratios : list, optional
        L1 ratio values to test. Default: [0.1, 0.3, 0.5, 0.7, 0.9]
    alphas : list, optional
        Alpha values to test. Default: [0.001, 0.01, 0.1, 1.0, 10.0]
        
    Returns
    -------
    dict
        Dictionary with keys:
        - 'model': fitted ElasticNet model
        - 'best_l1_ratio': best l1 ratio
        - 'best_alpha': best alpha
        - 'train_r2': R² on training data
        - 'test_r2': R² on test data
        - 'results_df': full results DataFrame
    """    
    # TODO: Implement best model selection
    # - Train models using train_elasticnet_grid
    # - Select model with highest test R² (not training R²)
    # - Return dictionary with best model and parameters

    if l1_ratios is None:
        l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
    if alphas is None:
        alphas = [0.001, 0.01, 0.1, 1.0, 10.0]
    
    results_df = train_elasticnet_grid(X_train, y_train, l1_ratios, alphas)
    
    testingr2 = []
    for _, row in results_df.iterrows():
        model = row["model"]
        test_r2 = r2_score(y_test, model.predict(X_test))
        testingr2.append(test_r2)

    results_df["test_r2"] = testingr2

    best_i = results_df["test_r2"].idxmax()
    best_row = results_df.loc[best_i]

    best_model = best_row["model"]
    best_l1 = best_row["l1_ratio"]
    best_alpha = best_row["alpha"]
    best_train_r2 = best_row["r2_score"]
    best_test_r2 = best_row["test_r2"]

    return {
        "model": best_model,
        "best_l1_ratio": best_l1,
        "best_alpha": best_alpha,
        "train_r2": best_train_r2,
        "test_r2": best_test_r2,
        "results_df": results_df
    }

get_best_elasticnet_model(X_train_scaled, y_train, X_test_scaled, y_test)

{'model': ElasticNet(alpha=0.01, l1_ratio=0.1, max_iter=5000, random_state=42),
 'best_l1_ratio': np.float64(0.1),
 'best_alpha': np.float64(0.01),
 'train_r2': np.float64(0.5093489767884205),
 'test_r2': np.float64(0.47888883023114626),
 'results_df':     l1_ratio   alpha  r2_score  \
 0        0.1   0.001  0.509453   
 1        0.1   0.010  0.509349   
 2        0.1   0.100  0.501134   
 3        0.1   1.000  0.235091   
 4        0.1  10.000  0.000000   
 5        0.3   0.001  0.509445   
 6        0.3   0.010  0.508995   
 7        0.3   0.100  0.472288   
 8        0.3   1.000  0.000000   
 9        0.3  10.000  0.000000   
 10       0.5   0.001  0.509439   
 11       0.5   0.010  0.508361   
 12       0.5   0.100  0.433767   
 13       0.5   1.000  0.000000   
 14       0.5  10.000  0.000000   
 15       0.7   0.001  0.509429   
 16       0.7   0.010  0.507489   
 17       0.7   0.100  0.389076   
 18       0.7   1.000  0.000000   
 19       0.7  10.000  0.000000   
 20       0.9

Best L1 ratio was 0.1, meaning it uses much more L2 Ridge (0) than L1 lasso (1), which akes sense when features are correlated like this heart disease dataset likely has.
Best alpha was 0.01, which means just light regularization was better.
The training r2 is only slightly higher than the test which is good and a sign of minimal overfitting. Values are around 0.5, which is decent.